<a href="https://colab.research.google.com/github/racoope70/daytrading-with-ml/blob/main/lightgbm_quantconnect_ready.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get remove --purge -y cuda* libcuda* nvidia* || echo "No conflicting CUDA packages"
!apt-get autoremove -y
!apt-get clean

In [ ]:
#Protocol Buffer Fix (for TensorFlow)
!pip uninstall -y protobuf
!pip install protobuf==3.20.3

In [ ]:
#Update Colab Environment and System Libraries
!apt-get update -y && apt-get upgrade -y


In [ ]:
#Install Correct Version of CUDA for Colab GPU
!apt-get update -qq && apt-get install -y \
    libcusolver11 libcusparse11 libcurand10 libcufft10 libnppig10 libnppc10 libnppial10 \
    cuda-toolkit-12-4

In [ ]:
#Set Correct CUDA Paths
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda-12.4'
os.environ['PATH'] += ':/usr/local/cuda-12.4/bin'
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-12.4/lib64'


In [ ]:
#Install RAPIDS and NVIDIA Dependencies
!pip install --extra-index-url=https://pypi.nvidia.com \
    cuml-cu12==25.2.0 cudf-cu12==25.2.0 cupy-cuda12x dask-cuda==25.2.0 dask-cudf-cu12==25.2.0


In [ ]:
#Install TensorFlow (latest GPU-compatible version)
!pip install tensorflow==2.18.0

#Install Stable Baselines3 and Trading Libraries
!pip install stable-baselines3[extra] gymnasium gym-anytrading yfinance xgboost joblib

#Install Miscellaneous Libraries
!pip install matplotlib scikit-learn pandas numba==0.61.0

#Install PyTorch with GPU Support
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124


In [ ]:
#Install TensorFlow (latest GPU-compatible version)
!pip install tensorflow==2.18.0


import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("TensorFlow GPU memory growth enabled")
    except RuntimeError as e:
        print(f"TensorFlow GPU memory config failed: {e}")


In [ ]:
!pip install stable-baselines3[extra] gymnasium gym-anytrading yfinance --quiet
!pip install stable-baselines3[extra] --quiet


In [ ]:
!rm -rf /content/drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)

In [ ]:
# === Imports ===
import os
import gc
import json
import time
import random
import joblib
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from datetime import datetime
from collections import deque

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from cuml.ensemble import RandomForestClassifier
import cupy as cp
import cudf
import cuml
import dask
import numba

import gymnasium as gym
import gym_anytrading
from gym_anytrading.envs import StocksEnv
from gymnasium.spaces import Box

from stable_baselines3 import PPO, SAC
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.vec_env import DummyVecEnv

# === Ticker List and Config ===
ticker_list = [
    'AAPL', 'TSLA', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'BRK-B', 'JPM', 'JNJ',
    'XOM', 'V', 'PG', 'UNH', 'MA', 'HD', 'LLY', 'MRK', 'PEP', 'KO',
    'BAC', 'ABBV', 'AVGO', 'PFE', 'COST', 'CSCO', 'TMO', 'ABT', 'ACN', 'WMT',
    'MCD', 'ADBE', 'DHR', 'CRM', 'NKE', 'INTC', 'QCOM', 'NEE', 'AMD', 'TXN',
    'AMGN', 'UPS', 'LIN', 'PM', 'UNP', 'BMY', 'LOW', 'RTX', 'CVX', 'IBM',
    'GE', 'SBUX', 'ORCL'
]

strategy_name = "sac_ppo_td3_multi_stock_v1"

CONFIG = {
    'symbols': [],
    'period': '720d',
    'interval': '1h',
    'target': 'Target',
    'sharpe_threshold': 1.5,
    'return_threshold': 1.25,
    'strategy_name': strategy_name
}

# === Download Function ===
def download_stock_data(ticker, period="720d", interval="1h", max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Attempt {attempt}: Downloading {ticker}...")
            df = yf.download(ticker, period=period, interval=interval)
            if not df.empty:
                df.reset_index(inplace=True)
                df['Symbol'] = ticker
                return df
            raise ValueError("Empty data")
        except Exception as e:
            print(f"Error: {e}. Retrying in {attempt * 5} sec...")
            time.sleep(attempt * 5)
    print(f"Failed to download {ticker}")
    return None

# === Feature Engineering Function ===
def compute_enhanced_features(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.loc[:, ~df.columns.duplicated()]

    df['SMA_20'] = df['Close'].rolling(20).mean()
    df['STD_20'] = df['Close'].rolling(20).std()
    df['Upper_Band'] = df['SMA_20'] + 2 * df['STD_20']
    df['Lower_Band'] = df['SMA_20'] - 2 * df['STD_20']
    df['Lowest_Low'] = df['Low'].rolling(14).min()
    df['Highest_High'] = df['High'].rolling(14).max()
    denom = (df['Highest_High'] - df['Lowest_Low']).replace(0, np.nan)
    df['Stoch'] = ((df['Close'] - df['Lowest_Low']) / denom) * 100
    df['ROC'] = df['Close'].pct_change(10)
    df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).cumsum()
    typical_price = (df['High'] + df['Low'] + df['Close']) / 3
    df['CCI'] = (typical_price - typical_price.rolling(20).mean()) / (0.015 * typical_price.rolling(20).std())
    df['PROC'] = ((df['Close'] - df['Close'].shift(12)) / df['Close'].shift(12)) * 100
    df['Rolling_Mean_50'] = df['Close'].rolling(50).mean()
    df['Expanding_Mean'] = df['Close'].expanding().mean()
    df['EMA_10'] = df['Close'].ewm(span=10).mean()
    df['EMA_50'] = df['Close'].ewm(span=50).mean()
    df['MACD_Line'] = df['Close'].ewm(span=12).mean() - df['Close'].ewm(span=26).mean()
    df['MACD_Signal'] = df['MACD_Line'].ewm(span=9).mean()
    df['MACD_Hist'] = df['MACD_Line'] - df['MACD_Signal']
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    df['True_Range'] = df[['High', 'Low', 'Close']].apply(
        lambda x: max(x.iloc[0] - x.iloc[1], abs(x.iloc[0] - x.iloc[2]), abs(x.iloc[1] - x.iloc[2])), axis=1)
    df['ATR'] = df['True_Range'].rolling(14).mean()
    df['+DM'] = np.where((df['High'].diff() > df['Low'].diff()) & (df['High'].diff() > 0), df['High'].diff(), 0)
    df['-DM'] = np.where((df['Low'].diff() > df['High'].diff()) & (df['Low'].diff() > 0), df['Low'].diff(), 0)
    df['+DI'] = 100 * df['+DM'].rolling(14).mean() / df['ATR']
    df['-DI'] = 100 * df['-DM'].rolling(14).mean() / df['ATR']
    df['ADX'] = abs(df['+DI'] - df['-DI']).rolling(14).mean()
    df['Volume_Avg'] = df['Volume'].rolling(20).mean()
    df['Volume_Change'] = df['Volume'].pct_change()
    df['Volume_Change_MA'] = df['Volume_Change'].rolling(10).mean()
    df['Volume_Change_Ratio'] = df['Volume_Change'] / df['Volume_Change'].shift(1)
    df['Relative_Volume'] = df['Volume'] / df['Volume_Avg']
    df['Trailing_Stop'] = np.minimum(df['Close'] * 0.985, df['Close'] - (df['ATR'] * 0.3))
    df['Buy_Signal'] = np.where((df['RSI'] < 60) & (df['EMA_10'] > df['EMA_50']) &
                                ((df['MACD_Line'] > df['MACD_Signal']) | (df['MACD_Line'].diff() > 0)) &
                                (df['Volume'] > (0.4 * df['Volume_Avg'])) & (df['ADX'] > 18), 1, 0)
    df['Sell_Signal'] = np.where(((df['EMA_10'] < df['EMA_50']) & (df['RSI'] > 60)) |
                                 ((df['MACD_Line'] < df['MACD_Signal']) & (df['RSI'] > 65)) |
                                 (df['Close'] < df['Trailing_Stop']) |
                                 ((df['Volume'] > 0.5 * df['Volume_Avg']) & (df['ADX'] > 20)), 1, 0)
    df['Sell_Signal_Debug'] = np.where(((df['MACD_Hist'] < 0.5) | (df['MACD_Line'] < df['MACD_Signal'])) &
                                       (df['RSI'] < 55) & (df['ADX'] > 15) &
                                       ((df['Close'] < df['Trailing_Stop']) | (df['EMA_10'] < df['EMA_50'])) &
                                       (df['Volume'] > 0.5 * df['Volume_Avg']), 1, 0)
    df['Future_Close'] = df['Close'].shift(-10)
    df['Volatility'] = df['Close'].pct_change().rolling(window=20).std()
    df['Return'] = (df['Future_Close'] - df['Close']) / df['Close']
    df['Target'] = np.select([df['Return'] > 0.02, df['Return'] < -0.02], [1, -1], default=0)
    df['Multi_Class_Target'] = df['Target']
    df['Hour'] = pd.to_datetime(df['Datetime']).dt.hour
    df['DayOfWeek'] = pd.to_datetime(df['Datetime']).dt.dayofweek
    df['Session'] = np.where((df['Hour'] >= 9) & (df['Hour'] <= 16), 'Regular',
                             np.where((df['Hour'] < 9), 'Pre-market', 'After-hours'))
    df['MACD_Crossover'] = np.where(df['MACD_Line'] > df['MACD_Signal'], 1, 0)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    return df

# === Main Pipeline ===
all_dfs = []

for ticker in ticker_list:
    df_single = download_stock_data(ticker, period=CONFIG['period'], interval=CONFIG['interval'])
    if df_single is not None:
        try:
            df_features = compute_enhanced_features(df_single)
            all_dfs.append(df_features)
        except Exception as e:
            print(f"Feature engineering failed for {ticker}: {e}")
    else:
        print(f"Failed to download {ticker}")

if all_dfs:
    df = pd.concat(all_dfs, ignore_index=True)
    print(f"Combined dataset created with shape: {df.shape}")
else:
    df = pd.DataFrame()
    print("No data available.")

if not df.empty:
    df.to_csv("multi_stock_feature_engineered_dataset.csv", index=False)
    print("Saved locally to multi_stock_feature_engineered_dataset.csv")

    drive_path = "/content/drive/MyDrive/trading_data/"
    os.makedirs(drive_path, exist_ok=True)
    df.to_csv(os.path.join(drive_path, "multi_stock_feature_engineered_dataset.csv"), index=False)
    print(f"Also saved to Google Drive at {drive_path}multi_stock_feature_engineered_dataset.csv")


Combined dataset created with shape: (262328, 51)
Saved locally to multi_stock_feature_engineered_dataset.csv
Also saved to Google Drive at /content/drive/MyDrive/trading_data/multi_stock_feature_engineered_dataset.csv


In [ ]:
# === Imports ===
import os
import gc
import time
import json
import pickle
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from lightgbm import LGBMClassifier
from lightgbm import early_stopping, log_evaluation

# === Paths ===
SAVE_DIR = "/content/drive/MyDrive/QuantConnect/results_lightgbm/lightgbm_walkforward_models"
RESULTS_DIR = "/content/drive/MyDrive/QuantConnect/results_lightgbm"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "data"), exist_ok=True)

# === Configuration ===
TEST_MODE = False
TICKERS = ['AAPL'] if TEST_MODE else [
    'AAPL', 'TSLA', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'BRK-B', 'JPM', 'JNJ',
    'XOM', 'V', 'PG', 'UNH', 'MA', 'HD', 'LLY', 'MRK', 'PEP', 'KO',
    'BAC', 'ABBV', 'AVGO', 'PFE', 'COST', 'CSCO', 'TMO', 'ABT', 'ACN', 'WMT',
    'MCD', 'ADBE', 'DHR', 'CRM', 'NKE', 'INTC', 'QCOM', 'NEE', 'AMD', 'TXN',
    'AMGN', 'UPS', 'LIN', 'PM', 'UNP', 'BMY', 'LOW', 'RTX', 'CVX', 'IBM',
    'GE', 'SBUX', 'ORCL']
features_list = ["SMA_50", "EMA_20", "RSI", "MACD", "Signal_Line", "ATR", "OBV", "CCI"]

# === Feature Engineering ===
def compute_technical_indicators(df):
    df['SMA_50'] = df['Close'].rolling(window=50).mean()
    df['EMA_20'] = df['Close'].ewm(span=20).mean()
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(window=14).mean()
    loss = -delta.where(delta < 0, 0).rolling(window=14).mean()
    rs = gain / (loss + 1e-6)
    df['RSI'] = 100 - (100 / (1 + rs))
    df['MACD'] = df['Close'].ewm(span=12).mean() - df['Close'].ewm(span=26).mean()
    df['Signal_Line'] = df['MACD'].ewm(span=9).mean()
    df['ATR'] = df['High'].rolling(window=14).max() - df['Low'].rolling(window=14).min()
    df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()
    tp = (df['High'] + df['Low'] + df['Close']) / 3
    df['CCI'] = (tp - tp.rolling(20).mean()) / (0.015 * tp.rolling(20).std())
    df.dropna(inplace=True)
    return df

# === Label Generation ===
def generate_labels(df):
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    return df.dropna()

# === Save Scaler and Feature Metadata ===
def save_scaler_and_features(df, ticker):
    scaler = MinMaxScaler()
    scaler.fit(df[features_list])
    with open(os.path.join(SAVE_DIR, f"scaler_{ticker}.pkl"), "wb") as f:
        pickle.dump(scaler, f, protocol=4)
    with open(os.path.join(SAVE_DIR, f"features_{ticker}.json"), "w") as f:
        json.dump(features_list, f)
    print(f" Saved scaler and features for {ticker}")

# === Walkforward Training + Evaluation ===
def run_lgbm_walkforward(ticker, window_size=3000, step_size=500, initial_cash=100000):
    print(f"\n Running walkforward LGBM for {ticker}")
    df = yf.download(ticker, period="720d", interval="1h", progress=False)
    if df.empty:
        print(" Empty dataframe. Skipping.")
        return

    df = compute_technical_indicators(df)
    df = generate_labels(df)
    if df.empty or not all(col in df.columns for col in features_list):
        print(" Missing required columns after feature engineering.")
        return

    save_scaler_and_features(df, ticker)

    scaler = MinMaxScaler()
    portfolio_lgbm, portfolio_hold, y_pred_all, y_true_all = [], [], [], []
    capital = initial_cash
    shares = initial_cash / df['Close'].iloc[0]

    for start in range(0, len(df) - window_size, step_size):
        segment = df.iloc[start:start+window_size].copy()
        if segment.shape[0] < window_size:
            break

        X = segment[features_list]
        y = segment['Target']
        X_scaled = scaler.fit_transform(X)
        X_train, X_test = X_scaled[:-step_size], X_scaled[-step_size:]
        y_train, y_test = y[:-step_size], y[-step_size:]

        model = LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42)
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  callbacks=[early_stopping(20), log_evaluation(0)])

        preds = (model.predict(X_test) > 0.5).astype(int)
        prices = segment['Close'].values[-step_size:]

        if len(preds) != step_size or len(prices) != step_size:
            continue

        y_pred_all.extend(preds)
        y_true_all.extend(y_test.tolist())

        for i, pred in enumerate(preds):
            price = prices[i]
            if pred:
                capital = shares * price
            else:
                shares = capital / price
            portfolio_lgbm.append(float(capital))
            portfolio_hold.append(float(shares * price))

    if len(y_pred_all) < 2:
        print(f" Skipping {ticker} — not enough data for evaluation.")
        return

    y_pred_all = np.array(y_pred_all[:len(y_true_all)])
    y_true_all = np.array(y_true_all[:len(y_pred_all)])

    accuracy = accuracy_score(y_true_all, y_pred_all)
    precision = precision_score(y_true_all, y_pred_all)
    recall = recall_score(y_true_all, y_pred_all)
    f1 = f1_score(y_true_all, y_pred_all)
    returns = np.diff(portfolio_lgbm) / (np.array(portfolio_lgbm[:-1]) + 1e-6)
    sharpe = np.mean(returns) / (np.std(returns) + 1e-6) * np.sqrt(252)
    drawdown = np.max(np.maximum.accumulate(portfolio_lgbm) - portfolio_lgbm)

    final_portfolio = portfolio_lgbm[-1]
    final_hold = portfolio_hold[-1]

    print(f" {ticker} | Final: {final_portfolio:.2f} | Buy&Hold: {final_hold:.2f} | Sharpe: {sharpe:.3f} | Accuracy: {accuracy:.3f}")

    model.booster_.save_model(f"{SAVE_DIR}/model_{ticker}.txt")

    metrics_df = pd.DataFrame([{
        "Ticker": ticker,
        "Final_Portfolio": round(final_portfolio, 2),
        "Final_Hold": round(final_hold, 2),
        "Return_%": round((final_portfolio - initial_cash) / initial_cash * 100, 2),
        "Return_Hold_%": round((final_hold - initial_cash) / initial_cash * 100, 2),
        "Sharpe": round(sharpe, 4),
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1_Score": round(f1, 4),
        "Drawdown": round(drawdown, 2)
    }])
    metrics_df.to_csv(f"{RESULTS_DIR}/metrics_{ticker}.csv", index=False)

    summary_path = os.path.join(RESULTS_DIR, "lightgbm_walkforward_summary.csv")
    if os.path.exists(summary_path):
        all_metrics = pd.read_csv(summary_path)
        all_metrics = all_metrics[all_metrics["Ticker"] != ticker]
        all_metrics = pd.concat([all_metrics, metrics_df], ignore_index=True)
    else:
        all_metrics = metrics_df
    all_metrics.to_csv(summary_path, index=False)

    plt.figure(figsize=(12, 6))
    plt.plot(portfolio_lgbm, label="LGBM Strategy")
    plt.plot(portfolio_hold, label="Buy & Hold")
    plt.title(f"{ticker} Portfolio Value")
    plt.xlabel("Step")
    plt.ylabel("Portfolio Value")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/plots/{ticker}_portfolio_plot.png")
    plt.close()

    print(f" Saved all artifacts for {ticker}")

# === Run All Tickers ===
for ticker in TICKERS:
    run_lgbm_walkforward(ticker)
    gc.collect()
    time.sleep(1)


AAPL | Final: 107813.50 | Buy&Hold: 107813.50 | Sharpe: -0.001 | Accuracy: 0.518
TSLA | Final: 60003.96 | Buy&Hold: 60003.96 | Sharpe: 0.015 | Accuracy: 0.519
MSFT | Final: 109526.73 | Buy&Hold: 109526.73 | Sharpe: -0.296 | Accuracy: 0.515
GOOGL | Final: 94624.64 | Buy&Hold: 94624.64 | Sharpe: -0.265 | Accuracy: 0.505
AMZN | Final: 111882.72 | Buy&Hold: 111882.72 | Sharpe: 0.084 | Accuracy: 0.514
NVDA | Final: 420776.81 | Buy&Hold: 420776.81 | Sharpe: 0.266 | Accuracy: 0.522
META | Final: 323565.58 | Buy&Hold: 323565.58 | Sharpe: 0.497 | Accuracy: 0.516
BRK-B | Final: 169447.16 | Buy&Hold: 169447.16 | Sharpe: 0.710 | Accuracy: 0.525
JPM | Final: 190284.42 | Buy&Hold: 190284.42 | Sharpe: 0.494 | Accuracy: 0.539
JNJ | Final: 94143.62 | Buy&Hold: 94143.62 | Sharpe: -0.037 | Accuracy: 0.529
XOM | Final: 116740.28 | Buy&Hold: 116740.28 | Sharpe: 0.214 | Accuracy: 0.524
V | Final: 155434.71 | Buy&Hold: 155434.71 | Sharpe: 0.416 | Accuracy: 0.514
PG | Final: 99048.64 | Buy&Hold: 99048.64 | Sh